In [1]:
import numpy as np
import pandas as pd

# Зафиксируем seed
SEED = 42
np.random.seed(SEED)

# Параметры
N_ORDERS = 15_000           # заказов (не строк!)
ITEMS_PER_ORDER_MEAN = 3.3  # → ~50k строк
MAX_ITEMS = 8

N_USERS = 5_000
N_PRODUCTS = 200
N_INGREDIENTS = 50
N_WORDS = 100
N_CATEGORIES = 5

# Словари
user_ids = [f"user_{i+1:04d}" for i in range(N_USERS)]
product_names = [f"название_{i+1}" for i in range(N_PRODUCTS)]
ingredients = [f"ингрид_{i+1}" for i in range(N_INGREDIENTS)]
words = [f"слово_{i+1}" for i in range(N_WORDS)]
categories = [f"category_{i+1}" for i in range(N_CATEGORIES)]
cat_to_id = {cat: i+1 for i, cat in enumerate(categories)}

print("✅ Шаг 0 завершён: импорты, seed, словари")

✅ Шаг 0 завершён: импорты, seed, словари


In [2]:
# --- 1. Генерация времён заказов (N_ORDERS штук) ---
order_datetimes = []
order_user_ids = []

dates_2025 = pd.date_range("2025-01-01", "2025-12-31", freq="D")
daily_counts = np.random.randint(30, 55, size=len(dates_2025))
daily_counts = (daily_counts * N_ORDERS / daily_counts.sum()).astype(int)
diff = N_ORDERS - daily_counts.sum()
daily_counts[:abs(diff)] += np.sign(diff)

for day, n in zip(dates_2025, daily_counts):
    for _ in range(n):
        h = np.random.randint(7, 23)
        m = np.random.randint(0, 60)
        s = np.random.randint(0, 60)
        dt = pd.Timestamp(day) + pd.Timedelta(hours=h, minutes=m, seconds=s)
        order_datetimes.append(dt)
        order_user_ids.append(np.random.choice(user_ids))

print(f"✅ Сгенерировано заказов: {len(order_datetimes)}")

# --- 2. Раскладываем каждый заказ на 1–8 товаров ---
rows = []
for dt, uid in zip(order_datetimes, order_user_ids):
    n_items = np.random.randint(1, MAX_ITEMS + 1)
    if np.random.rand() < 0.3:
        n_items = max(2, n_items)  # поднимаем хвост
    
    for _ in range(n_items):
        rows.append({
            "datetime": dt,
            "user_id": uid,
            "product_id": f"prod_{np.random.randint(1, N_PRODUCTS+1):04d}",
            "product_name": np.random.choice(product_names),
            "product_category": np.random.choice(categories, p=[0.3, 0.25, 0.2, 0.15, 0.1]),
        })

df = pd.DataFrame(rows)
print(f"✅ Строк (товаров): {len(df)}")
print(f"   уникальных заказов (уникальных datetime): {df['datetime'].nunique()}")
print(f"   товаров в заказе: min={df.groupby('datetime').size().min()}, "
      f"med={df.groupby('datetime').size().median():.1f}, "
      f"max={df.groupby('datetime').size().max()}")

✅ Сгенерировано заказов: 15000
✅ Строк (товаров): 68762
   уникальных заказов (уникальных datetime): 14998
   товаров в заказе: min=1, med=5.0, max=12


In [3]:
# Добавим уникальный order_id по уникальному datetime + user_id (на случай совпадений)
df = df.sort_values(["datetime", "user_id"]).reset_index(drop=True)
df["order_id"] = (
    df["datetime"].astype(str) + "_" + df["user_id"]
).factorize()[0] + 1
df["order_id"] = "order_" + df["order_id"].astype(str).str.zfill(6)

# --- Базовые признаки на уровне товара ---
df["category_id"] = df["product_category"].map(cat_to_id)

# ингредиенты (1–4)
df["ингредиенты"] = [
    " ".join(np.random.choice(ingredients, size=np.random.randint(1, 5), replace=False))
    for _ in range(len(df))
]

# вес_товара_кг (0.05–10 кг)
weights = np.round(np.random.exponential(1.2, len(df)) + 0.2, 2)
df["вес_товара_кг"] = np.clip(weights, 0.05, 10.0)

# стоимость_товара (на основе веса, категории, шума)
base_price_per_kg = np.array([300, 450, 650, 900, 1200])  # category_1 ... category_5
price_per_kg = df["category_id"].map({i+1: base_price_per_kg[i] for i in range(5)}).values
df["стоимость_товара"] = np.round(
    df["вес_товара_кг"] * price_per_kg * np.random.uniform(0.8, 1.2, len(df)),
    2
)

# --- Бинарные флаги (на уровне заказа → реплицируем на все товары в заказе) ---
# Сначала сгенерируем для уникальных заказов
order_flags = {}
unique_orders = df[["order_id", "datetime", "user_id"]].drop_duplicates().copy()
n_orders = len(unique_orders)

# Генерация для каждого заказа
np.random.seed(SEED + 1)  # новый seed для воспроизводимости
order_flags["позвонить_если_нет_в_наличии"] = np.random.binomial(1, 0.45, n_orders)
order_flags["оставить_у_двери_и_позвонить"] = np.random.binomial(1, 0.35, n_orders)
order_flags["заказ_примет_другой_человек"] = np.random.binomial(1, 0.12, n_orders)
order_flags["без_чаевых_курьеру"] = np.random.binomial(1, 0.20, n_orders)
order_flags["без_чаевых_сборщику"] = np.random.binomial(1, 0.25, n_orders)
order_flags["промокод"] = np.random.binomial(1, 0.25, n_orders)
order_flags["купон"] = np.random.binomial(1, 0.18, n_orders)

# Присоединяем к df
for col, values in order_flags.items():
    mapping = dict(zip(unique_orders["order_id"], values))
    df[col] = df["order_id"].map(mapping)

print("✅ Добавлены: order_id, категории, ингредиенты, вес, стоимость_товара, флаги заказа")
print(f"   уникальных заказов: {df['order_id'].nunique()}")
print(f"   средняя стоимость товара: {df['стоимость_товара'].mean():.2f} руб")
print(f"   пример стоимости: {df['стоимость_товара'].iloc[:5].tolist()}")

✅ Добавлены: order_id, категории, ингредиенты, вес, стоимость_товара, флаги заказа
   уникальных заказов: 15000
   средняя стоимость товара: 820.11 руб
   пример стоимости: [144.89, 2529.54, 139.86, 1378.76, 254.27]


In [4]:
# Агрегация по order_id
order_agg = df.groupby("order_id").agg(
    datetime=("datetime", "first"),
    user_id=("user_id", "first"),
    количество_товаров=("product_id", "count"),
    сумма_заказа=("стоимость_товара", "sum"),
    вес_заказа_кг=("вес_товара_кг", "sum")
).reset_index()

# Применяем скидки на уровне заказа (промокод = -10%, купон = -100 руб)
# Получим флаги заказа обратно
flag_cols = [
    "позвонить_если_нет_в_наличии", "оставить_у_двери_и_позвонить",
    "заказ_примет_другой_человек", "без_чаевых_курьеру",
    "без_чаевых_сборщику", "промокод", "купон"
]
for col in flag_cols:
    order_agg[col] = df.drop_duplicates("order_id").set_index("order_id")[col].reindex(order_agg["order_id"]).values

# Применяем скидки
order_agg["сумма_заказа"] = order_agg["сумма_заказа"] * (0.9 ** order_agg["промокод"])
order_agg["сумма_заказа"] = order_agg["сумма_заказа"] - 100 * order_agg["купон"]
order_agg["сумма_заказа"] = np.round(np.clip(order_agg["сумма_заказа"], 50, 5000), 2)

print("✅ Агрегаты по заказу вычислены")
print(f"   средний чек: {order_agg['сумма_заказа'].mean():.2f} руб")
print(f"   медианный чек: {order_agg['сумма_заказа'].median():.2f} руб")
print(f"   пример сумм заказов: {order_agg['сумма_заказа'].iloc[:5].tolist()}")

✅ Агрегаты по заказу вычислены
   средний чек: 3085.79 руб
   медианный чек: 3176.55 руб
   пример сумм заказов: [5000.0, 4482.1, 5000.0, 5000.0, 2241.97]


In [5]:
# Генерация текстов для заказа (реплицируем на все товары)
def gen_comment(max_words, p_empty=0.0):
    if np.random.rand() < p_empty:
        return ""
    n = np.random.randint(0, max_words + 1)
    return " ".join(np.random.choice(words, n, replace=False)) if n > 0 else ""

np.random.seed(SEED + 2)

# Добавим вероятность пустого отзыва: 40% не оставляют
p_empty_review = 0.40

order_agg["коммент_сборщику"] = [gen_comment(3, p_empty=0.25) for _ in range(len(order_agg))]
order_agg["коммент_курьеру"] = [gen_comment(2, p_empty=0.35) for _ in range(len(order_agg))]
order_agg["рукописная_обратная_связь"] = [
    gen_comment(6, p_empty=p_empty_review) for _ in range(len(order_agg))
]

# Присоединяем к df
text_cols = ["коммент_сборщику", "коммент_курьеру", "рукописная_обратная_связь"]
for col in text_cols:
    mapping = dict(zip(order_agg["order_id"], order_agg[col]))
    df[col] = df["order_id"].map(mapping)

print("✅ Комментарии добавлены (с пропусками)")
print(f"   пустых отзывов: {(df['рукописная_обратная_связь'] == '').sum():,} ({(df['рукописная_обратная_связь'] == '').mean():.1%})")
print("🔍 Примеры:")
for i in range(3):
    print(f"  {i+1}) коммент_сборщику: '{df.iloc[i]['коммент_сборщику']}'")
    print(f"     коммент_курьеру:   '{df.iloc[i]['коммент_курьеру']}'")
    print(f"     отзыв:             '{df.iloc[i]['рукописная_обратная_связь'][:50]}...'")

✅ Комментарии добавлены (с пропусками)
   пустых отзывов: 33,464 (48.7%)
🔍 Примеры:
  1) коммент_сборщику: 'слово_55'
     коммент_курьеру:   'слово_4'
     отзыв:             '...'
  2) коммент_сборщику: 'слово_55'
     коммент_курьеру:   'слово_4'
     отзыв:             '...'
  3) коммент_сборщику: 'слово_55'
     коммент_курьеру:   'слово_4'
     отзыв:             '...'


In [6]:
# Генерация оценки заказа (1–5) — но не у всех
np.random.seed(SEED + 3)

# 60% ставят оценку, 40% — пропускают
p_has_rating = 0.60
has_rating = np.random.binomial(1, p_has_rating, len(order_agg)).astype(bool)

# Распределение только для тех, кто оценил
probs = [0.03, 0.07, 0.15, 0.30, 0.45]
оценка_заказа = np.full(len(order_agg), np.nan)
оценка_заказа[has_rating] = np.random.choice([1, 2, 3, 4, 5], size=has_rating.sum(), p=probs)

# Сохраняем
order_agg["оценка_заказа"] = оценка_заказа

# Генерация галочек ТОЛЬКО для тех, кто поставил оценку
def gen_flag(score, p_low, p_high):
    p = p_low + (p_high - p_low) * (score - 1) / 4
    return np.random.binomial(1, p)

# Инициализируем как NaN
учли = np.full(len(order_agg), np.nan)
идеально = np.full(len(order_agg), np.nan)
быстро = np.full(len(order_agg), np.nan)
отличный = np.full(len(order_agg), np.nan)

# Заполняем только где есть оценка
score_vals = оценка_заказа[has_rating]
учли[has_rating] = gen_flag(score_vals, p_low=0.2, p_high=0.92)
идеально[has_rating] = gen_flag(score_vals, p_low=0.15, p_high=0.88)
быстро[has_rating] = gen_flag(score_vals, p_low=0.25, p_high=0.80)
отличный[has_rating] = gen_flag(score_vals, p_low=0.2, p_high=0.85)

# Добавляем
order_agg["учли_все_пожелания"] = учли
order_agg["идеально_собрали_заказ"] = идеально
order_agg["быстро_привезли"] = быстро
order_agg["отличный_курьер"] = отличный

# Мержим в df
for col in ["оценка_заказа", "учли_все_пожелания", "идеально_собрали_заказ", "быстро_привезли", "отличный_курьер"]:
    mapping = dict(zip(order_agg["order_id"], order_agg[col]))
    df[col] = df["order_id"].map(mapping)

print("\n✅ Оценка и галочки добавлены (с пропусками)")
print(f"   оценка_заказа (не NaN): {df['оценка_заказа'].notnull().sum():,} ({df['оценка_заказа'].notnull().mean():.1%})")
print(f"   учли_все_пожелания (не NaN): {df['учли_все_пожелания'].notnull().sum():,} ({df['учли_все_пожелания'].notnull().mean():.1%})")

# Проверка зависимости
for s in [1, 3, 5]:
    mask = (df["оценка_заказа"] == s) & df["учли_все_пожелания"].notnull()
    if mask.sum() > 0:
        rate = df.loc[mask, "учли_все_пожелания"].mean()
        print(f"  при оценке {s}★: учли=1 в {rate:.1%} случаев")


✅ Оценка и галочки добавлены (с пропусками)
   оценка_заказа (не NaN): 41,223 (60.0%)
   учли_все_пожелания (не NaN): 41,223 (60.0%)
  при оценке 1★: учли=1 в 21.1% случаев
  при оценке 3★: учли=1 в 56.7% случаев
  при оценке 5★: учли=1 в 91.8% случаев


In [7]:
# --- product_rating (внутренний рейтинг товара) ---
# Основан на оценке заказа + шум, но только там, где оценка есть
np.random.seed(SEED + 4)

# Создаём копию с NaN по умолчанию
product_rating = pd.Series(np.nan, index=df.index)

# Маска: где есть оценка
has_rating = df["оценка_заказа"].notnull()

# Добавляем шум и округляем → оставляем как float (чтобы NaN сохранялись)
rating_vals = df.loc[has_rating, "оценка_заказа"].values
noise = np.random.normal(0, 0.6, size=len(rating_vals))
product_rating.loc[has_rating] = np.round(rating_vals + noise)

# Ограничиваем 1–5, сохраняя NaN
product_rating = np.clip(product_rating, 1, 5)

df["product_rating"] = product_rating  # float64, с NaN

# --- Чаевые (исправлены: заменяем NaN в оценке на 3.0 для расчёта, но не меняем исходные данные)
# Для расчёта чаевых используем заполненную оценку (без изменения df["оценка_заказа"])
оценка_filled = df["оценка_заказа"].fillna(3.0)  # нейтральное значение
отличный_filled = df["отличный_курьер"].fillna(0.5)
идеально_filled = df["идеально_собрали_заказ"].fillna(0.5)

# Курьеру
tips_courier = np.where(
    df["без_чаевых_курьеру"] == 1,
    0,
    np.random.exponential(40, len(df)) * (0.2 + 0.1 * оценка_filled + 0.15 * отличный_filled)
)
df["сумма_чаевых_курьеру"] = np.round(np.clip(tips_courier, 0, 300), 2)

# Сборщику
tips_picker = np.where(
    df["без_чаевых_сборщику"] == 1,
    0,
    np.random.exponential(30, len(df)) * (0.1 + 0.12 * оценка_filled + 0.1 * идеально_filled)
)
df["сумма_чаевых_сборщику"] = np.round(np.clip(tips_picker, 0, 200), 2)

print("✅ product_rating и чаевые исправлены")

# --- Преобразуем в int64 (NaN → 0) ---
df["product_rating"] = df["product_rating"].fillna(0).astype("int64")
df["target_4"] = df["оценка_заказа"].fillna(0).astype("int64")

print(f"   product_rating: {df['product_rating'].value_counts().sort_index().to_dict()}")
print(f"   target_4: {df['target_4'].value_counts().sort_index().to_dict()}")
print(f"   чаевые курьеру (медиана >0): {df.loc[df['сумма_чаевых_курьеру']>0, 'сумма_чаевых_курьеру'].median():.2f} руб")

✅ product_rating и чаевые исправлены
   product_rating: {0: 27539, 1: 1567, 2: 3239, 3: 6964, 4: 12257, 5: 17196}
   target_4: {0: 27539, 1: 1165, 2: 2953, 3: 6325, 4: 12331, 5: 18449}
   чаевые курьеру (медиана >0): 17.70 руб


In [8]:
# --- 1. target_2, target_3, target_4 ---
df["target_2"] = df["product_category"]
# target_4 уже установлен выше: df["target_4"] = df["оценка_заказа"].fillna(0).astype("int64")
# target_3 = сумма_заказа (берём из order_agg)
sum_by_order = order_agg.set_index("order_id")["сумма_заказа"]
df["target_3"] = df["order_id"].map(sum_by_order)

# --- 2. target_1: повторная покупка в течение 7 дней ---
print("⏳ Вычисление target_1 (честно, по данным)...")

df_sorted = df.sort_values(["user_id", "datetime"]).copy()
df_sorted["date"] = df_sorted["datetime"].dt.date
df_sorted["target_1"] = 0

# Для каждого пользователя — проверяем окно +1ч ... +7дн
for user, group in df_sorted.groupby("user_id"):
    times = group["datetime"].values
    idxs = group.index.values
    for i, t in enumerate(times):
        window_end = t + pd.Timedelta(days=7)
        # Есть ли другие заказы этого пользователя в будущем (не включая сам заказ)?
        future = ((times > t + pd.Timedelta(hours=1)) & (times <= window_end)).sum()
        if future > 0:
            df_sorted.loc[idxs[i], "target_1"] = 1

df = df_sorted.reindex(df.index)

print("✅ target_1 вычислен")
print(f"target_1 = 1: {df['target_1'].mean():.2%} заказов")

# --- 3. Агрегаты по дню ---
daily_agg = df.groupby("date").agg(
    daily_order_count=("order_id", "nunique"),
    daily_total_revenue=("target_3", "first")  # так как target_3 одинаков в заказе — можно first/sum
).reset_index()

# Но правильно: сумма всех сумм заказов за день
daily_agg_correct = order_agg.copy()
daily_agg_correct["date"] = daily_agg_correct["datetime"].dt.date
daily_agg = daily_agg_correct.groupby("date").agg(
    daily_order_count=("order_id", "count"),
    daily_total_revenue=("сумма_заказа", "sum")
).reset_index()
daily_agg["daily_avg_check"] = daily_agg["daily_total_revenue"] / daily_agg["daily_order_count"]

# Мержим обратно в df
date_mapping = dict(zip(daily_agg["date"], daily_agg["daily_order_count"]))
df["daily_order_count"] = df["date"].map(date_mapping)

date_mapping = dict(zip(daily_agg["date"], daily_agg["daily_total_revenue"]))
df["daily_total_revenue"] = df["date"].map(date_mapping)

date_mapping = dict(zip(daily_agg["date"], daily_agg["daily_avg_check"]))
df["daily_avg_check"] = df["date"].map(date_mapping)

print("✅ Агрегаты по дню добавлены")
print(f"Среднее число заказов в день: {df['daily_order_count'].mean():.1f}")
print(f"Средняя выручка в день: {df['daily_total_revenue'].mean():,.0f} руб")

⏳ Вычисление target_1 (честно, по данным)...


✅ target_1 вычислен
target_1 = 1: 5.71% заказов
✅ Агрегаты по дню добавлены
Среднее число заказов в день: 42.5
Средняя выручка в день: 131,845 руб


In [9]:
# --- 4. Преобразуем галочки (target_5) в int64 (NaN → 0) ---
flag_cols = ["учли_все_пожелания", "идеально_собрали_заказ", "быстро_привезли", "отличный_курьер"]
for col in flag_cols:
    df[col] = df[col].fillna(0).astype("int64")

print("✅ Галочки (target_5) переведены в int64 (NaN → 0)")
print(f"   учли_все_пожелания: {df['учли_все_пожелания'].value_counts().sort_index().to_dict()}")
print(f"   идеально_собрали_заказ: {df['идеально_собрали_заказ'].value_counts().sort_index().to_dict()}")
print(f"   быстро_привезли: {df['быстро_привезли'].value_counts().sort_index().to_dict()}")
print(f"   отличный_курьер: {df['отличный_курьер'].value_counts().sort_index().to_dict()}")

✅ Галочки (target_5) переведены в int64 (NaN → 0)
   учли_все_пожелания: {0: 37735, 1: 31027}
   идеально_собрали_заказ: {0: 39386, 1: 29376}
   быстро_привезли: {0: 41023, 1: 27739}
   отличный_курьер: {0: 40254, 1: 28508}


In [10]:
print(df.head().to_markdown())

|    | datetime            | user_id   | product_id   | product_name   | product_category   | order_id     |   category_id | ингредиенты                             |   вес_товара_кг |   стоимость_товара |   позвонить_если_нет_в_наличии |   оставить_у_двери_и_позвонить |   заказ_примет_другой_человек |   без_чаевых_курьеру |   без_чаевых_сборщику |   промокод |   купон | коммент_сборщику   | коммент_курьеру   | рукописная_обратная_связь   |   оценка_заказа |   учли_все_пожелания |   идеально_собрали_заказ |   быстро_привезли |   отличный_курьер |   product_rating |   сумма_чаевых_курьеру |   сумма_чаевых_сборщику |   target_4 | target_2   |   target_3 | date       |   target_1 |   daily_order_count |   daily_total_revenue |   daily_avg_check |
|---:|:--------------------|:----------|:-------------|:---------------|:-------------------|:-------------|--------------:|:----------------------------------------|----------------:|-------------------:|-------------------------------:|--------

In [11]:
# --- Проверка target_1 (бинарная) ---
t1 = df["target_1"]
print("✅ target_1 (бинарная: покупка в течение 7 дней)")
print(f"   уникальные значения: {sorted(t1.unique())}")
print(f"   доля 1: {t1.mean():.2%}")
print(f"   проверка: все значения ∈ {{0,1}}? {set(t1.unique()).issubset({0, 1})}")

# --- Проверка target_2 (категориальная) ---
t2 = df["target_2"]
print("\n✅ target_2 (категориальная: категория товара)")
print(f"   уникальные значения: {sorted(t2.unique())}")
print(f"   кол-во классов: {t2.nunique()}")
print(f"   распределение:\n{t2.value_counts().sort_index()}")

# --- Проверка target_3 (регрессия: сумма заказа) ---
t3 = df["target_3"]
print("\n✅ target_3 (регрессия: сумма заказа)")
print(f"   min = {t3.min():.2f}, med = {t3.median():.2f}, max = {t3.max():.2f}")
print(f"   среднее = {t3.mean():.2f}")
print(f"   проверка: все > 0? {t3.min() > 0}")

# --- Проверка target_4 (порядковая: оценка заказа) ---
t4 = df["target_4"]
print("\n✅ target_4 (порядковая: оценка заказа, 1–5)")
print(f"   уникальные значения: {sorted(t4.unique())}")
print(f"   распределение:\n{t4.value_counts().sort_index()}")
print(f"   проверка: все ∈ {{0,1,2,3,4,5}}? {set(t4.unique()).issubset({0,1,2,3,4,5})}")

# --- Проверка target_5 (9 бинарных колонок) ---
target_5_cols = [
    "учли_все_пожелания",
    "идеально_собрали_заказ",
    "быстро_привезли",
    "отличный_курьер",
    "позвонить_если_нет_в_наличии",
    "оставить_у_двери_и_позвонить",
    "заказ_примет_другой_человек",
    "без_чаевых_курьеру",
    "без_чаевых_сборщику"
]

print("\n✅ target_5 (многометочная: 9 бинарных колонок)")
all_ok = True
for col in target_5_cols:
    vals = df[col].unique()
    ok = set(vals).issubset({0, 1})
    all_ok = all_ok and ok
    print(f"   {col:<30} → уникальные: {sorted(vals)} | бинарный? {ok}")

print(f"\n   ИТОГ: все 9 колонок бинарные? {all_ok}")

# --- Доп: убедимся, что target_3 = сумма_заказа (из order_agg) ---
sum_check = np.allclose(
    df.groupby("order_id")["стоимость_товара"].sum().reindex(df["order_id"]).values,
    df["target_3"].values,
    atol=0.02  # допуск на округление
)
print(f"\n✅ Проверка: target_3 = сумма(стоимость_товара) по заказу? {sum_check}")

✅ target_1 (бинарная: покупка в течение 7 дней)
   уникальные значения: [np.int64(0), np.int64(1)]
   доля 1: 5.71%
   проверка: все значения ∈ {0,1}? True

✅ target_2 (категориальная: категория товара)
   уникальные значения: [np.str_('category_1'), np.str_('category_2'), np.str_('category_3'), np.str_('category_4'), np.str_('category_5')]
   кол-во классов: 5
   распределение:
target_2
category_1    20681
category_2    17298
category_3    13658
category_4    10382
category_5     6743
Name: count, dtype: int64

✅ target_3 (регрессия: сумма заказа)
   min = 50.00, med = 4145.48, max = 5000.00
   среднее = 3700.44
   проверка: все > 0? True

✅ target_4 (порядковая: оценка заказа, 1–5)
   уникальные значения: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
   распределение:
target_4
0    27539
1     1165
2     2953
3     6325
4    12331
5    18449
Name: count, dtype: int64
   проверка: все ∈ {0,1,2,3,4,5}? True

✅ target_5 (многометочная: 9 бинарных колонок

In [12]:
# Проверим: target_3 = (сумма_товаров) * 0.9^промокод - 100*купон ?
order_costs = df.groupby("order_id")["стоимость_товара"].sum()
order_promo = df.drop_duplicates("order_id").set_index("order_id")["промокод"]
order_coupon = df.drop_duplicates("order_id").set_index("order_id")["купон"]

expected_target3 = order_costs * (0.9 ** order_promo) - 100 * order_coupon
expected_target3 = np.round(np.clip(expected_target3, 50, 5000), 2)

actual_target3 = df.drop_duplicates("order_id").set_index("order_id")["target_3"]

match = np.allclose(expected_target3, actual_target3, atol=0.02)
print(f"✅ target_3 = (сумма_товаров) * 0.9^промокод - 100*купон? {match}")
print(f"   пример расчёта для order_000001:")
print(f"     сумма товаров: {order_costs['order_000001']:.2f}")
print(f"     промокод: {order_promo['order_000001']}, купон: {order_coupon['order_000001']}")
print(f"     расчёт: {order_costs['order_000001']:.2f} * {0.9**order_promo['order_000001']:.1f} - {100*order_coupon['order_000001']:.0f} = {expected_target3['order_000001']:.2f}")
print(f"     target_3: {actual_target3['order_000001']:.2f}")

✅ target_3 = (сумма_товаров) * 0.9^промокод - 100*купон? True
   пример расчёта для order_000001:
     сумма товаров: 5856.95
     промокод: 0, купон: 1
     расчёт: 5856.95 * 1.0 - 100 = 5000.00
     target_3: 5000.00


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68762 entries, 0 to 68761
Data columns (total 36 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   datetime                      68762 non-null  datetime64[ns]
 1   user_id                       68762 non-null  object        
 2   product_id                    68762 non-null  object        
 3   product_name                  68762 non-null  object        
 4   product_category              68762 non-null  object        
 5   order_id                      68762 non-null  object        
 6   category_id                   68762 non-null  int64         
 7   ингредиенты                   68762 non-null  object        
 8   вес_товара_кг                 68762 non-null  float64       
 9   стоимость_товара              68762 non-null  float64       
 10  позвонить_если_нет_в_наличии  68762 non-null  int64         
 11  оставить_у_двери_и_позвонить

In [14]:
text_cols = [
    "ингредиенты",
    "коммент_сборщику",
    "коммент_курьеру",
    "рукописная_обратная_связь"
]

print("🔍 Анализ текстовых колонок:\n")

for col in text_cols:
    s = df[col].astype(str)  # на всякий случай
    
    # Статистика длины (в символах)
    lens = s.str.len()
    n_empty = (s == "").sum()
    
    # Проверка на NaN/None
    n_nan = s.isnull().sum()
    has_nan_str = (s == "nan").sum()
    has_none_str = (s == "None").sum()
    
    print(f"📌 {col}:")
    print(f"   всего строк: {len(s)}")
    print(f"   пустых (''): {n_empty:,} ({n_empty/len(s):.1%})")
    print(f"   NaN/None: {n_nan}")
    print(f"   'nan' как строка: {has_nan_str}, 'None' как строка: {has_none_str}")
    print(f"   длина символов → min: {lens.min()}, med: {lens.median():.0f}, max: {lens.max()}")
    
    # Примеры: 1 пустая, 1 короткая, 1 длинная
    empty_ex = s[s == ""].iloc[0] if n_empty > 0 else "—"
    short_ex = s[(lens > 0) & (lens <= 10)].iloc[0] if ((lens > 0) & (lens <= 10)).any() else "—"
    long_ex = s[lens == lens.max()].iloc[0]
    
    print(f"   примеры: пустая='{empty_ex}', короткая='{short_ex}', длинная='{long_ex[:50]}...'")
    print()

🔍 Анализ текстовых колонок:

📌 ингредиенты:
   всего строк: 68762
   пустых (''): 0 (0.0%)
   NaN/None: 0
   'nan' как строка: 0, 'None' как строка: 0
   длина символов → min: 8, med: 19, max: 39
   примеры: пустая='—', короткая='ингрид_48', длинная='ингрид_37 ингрид_46 ингрид_30 ингрид_35...'

📌 коммент_сборщику:
   всего строк: 68762
   пустых (''): 30,211 (43.9%)
   NaN/None: 0
   'nan' как строка: 0, 'None' как строка: 0
   длина символов → min: 0, med: 8, max: 27
   примеры: пустая='', короткая='слово_55', длинная='слово_17 слово_100 слово_69...'

📌 коммент_курьеру:
   всего строк: 68762
   пустых (''): 39,350 (57.2%)
   NaN/None: 0
   'nan' как строка: 0, 'None' как строка: 0
   длина символов → min: 0, med: 0, max: 18
   примеры: пустая='', короткая='слово_4', длинная='слово_42 слово_100...'

📌 рукописная_обратная_связь:
   всего строк: 68762
   пустых (''): 33,464 (48.7%)
   NaN/None: 0
   'nan' как строка: 0, 'None' как строка: 0
   длина символов → min: 0, med: 8, max: 54
   

In [15]:
df.to_csv("online_grocery_dataset.csv", index=False)

In [16]:
# Убедимся, что 'date' — формат datetime
df["date"] = pd.to_datetime(df["date"])

# --- Разделение ---
train_df = df[df["date"] < "2025-12-01"].copy()
test_df = df[df["date"] >= "2025-12-01"].copy()

print(f"✅ Размеры после разделения:")
print(f"   train_df: {len(train_df):,} строк")
print(f"   test_df:  {len(test_df):,} строк")
print(f"   дата в train: {train_df['date'].min()} — {train_df['date'].max()}")
print(f"   дата в test:  {test_df['date'].min()} — {test_df['date'].max()}")

# --- Сохранение ---
train_df.to_csv("train.csv", index=False)
test_df.to_csv("test.csv", index=False)

print("\n✅ Файлы сохранены: train.csv, test.csv")

✅ Размеры после разделения:
   train_df: 62,966 строк
   test_df:  5,796 строк
   дата в train: 2025-01-01 00:00:00 — 2025-11-30 00:00:00
   дата в test:  2025-12-01 00:00:00 — 2025-12-31 00:00:00

✅ Файлы сохранены: train.csv, test.csv
